# Lakehouse — Construir un Lakehouse Local con Delta Lake

## Unidad 4: Infraestructura de Datos

En los notebooks anteriores construimos un Data Warehouse (esquema estrella en SQLite) y un Data Lake (carpetas Bronze/Silver/Gold con Parquet). Cada uno tenia limitaciones: el Warehouse era rigido y no soportaba datos crudos, el Lake no tenia esquema ni transacciones.

En este notebook construimos un Lakehouse: usamos el mismo almacenamiento barato (archivos) pero le agregamos transacciones ACID, time travel y control de esquema con Delta Lake. Lo hacemos local, sin Spark, usando la libreria `deltalake` en Python.

### Contenido:
1. Setup: instalar y configurar
2. Crear tablas Delta (Bronze, Silver, Gold)
3. Schema enforcement: rechazar datos que no cumplen
4. Time travel: leer versiones anteriores
5. Merge/Upsert: actualizar registros existentes
6. Medallion Architecture completa con Delta
7. Consultas analiticas sobre Delta

In [1]:
# ============================================================
# INSTALACION
# ============================================================

# deltalake es la libreria Python nativa (sin Spark)
# Escrita en Rust, rapida y sin dependencias pesadas

# !pip install deltalake

import pandas as pd
import numpy as np
import os
import shutil
import json
from datetime import datetime

try:
    from deltalake import DeltaTable, write_deltalake
    DELTA_OK = True
    print("deltalake instalado correctamente")
except ImportError:
    DELTA_OK = False
    print("deltalake no instalado. Ejecuta: pip install deltalake")
    print("Los ejemplos se mostraran como referencia.")

deltalake no instalado. Ejecuta: pip install deltalake
Los ejemplos se mostraran como referencia.


In [2]:
# ============================================================
# ESTRUCTURA DEL LAKEHOUSE
# ============================================================

LAKEHOUSE = 'lakehouse'
if os.path.exists(LAKEHOUSE):
    shutil.rmtree(LAKEHOUSE)

for zona in ['bronze', 'silver', 'gold']:
    os.makedirs(f'{LAKEHOUSE}/{zona}', exist_ok=True)

print("Estructura creada:")
print(f"  {LAKEHOUSE}/")
print(f"    bronze/   ← Datos crudos como Delta (append-only, con historial)")
print(f"    silver/   ← Datos limpios como Delta (con schema enforcement)")
print(f"    gold/     ← Datos agregados como Delta (listos para BI y ML)")
print(f"\nLa diferencia con el Data Lake puro:")
print(f"  Cada zona es una TABLA DELTA, no solo una carpeta con archivos.")
print(f"  Eso agrega: ACID, time travel, esquema, merge.")

Estructura creada:
  lakehouse/
    bronze/   ← Datos crudos como Delta (append-only, con historial)
    silver/   ← Datos limpios como Delta (con schema enforcement)
    gold/     ← Datos agregados como Delta (listos para BI y ML)

La diferencia con el Data Lake puro:
  Cada zona es una TABLA DELTA, no solo una carpeta con archivos.
  Eso agrega: ACID, time travel, esquema, merge.


---
## 2. Crear tablas Delta

In [5]:
bronze_path = f'{LAKEHOUSE}/bronze/ventas'

for dia in ['2024-06-01', '2024-06-02', '2024-06-03']:
    n = np.random.randint(80, 150)
    df_dia = pd.DataFrame({
        'fecha': pd.Timestamp(dia),
        'producto': np.random.choice(['Dashboard', 'Reporte', 'API REST', 'App Web', 'Pipeline ETL'], n),
        'region': np.random.choice(['Bogota', 'Medellin', 'Cali', 'Manizales', 'Barranquilla'], n, p=[0.35, 0.25, 0.18, 0.12, 0.10]),
        'cliente': np.random.choice(['TechCorp', 'DataSoft', 'Analitika', 'InfoSystems', 'CloudBI', 'MetricaPro', 'VisionData'], n),
        'unidades': np.random.randint(1, 30, n),
        'precio': np.round(np.random.uniform(200, 800, n), 2),
        'ingesta_timestamp': datetime.now().isoformat(),  # Cuando llego al Lake
    })

    # Calculate mode here so it's always defined
    mode = 'overwrite' if dia == '2024-06-01' else 'append'

    if DELTA_OK:
        # Cada dia se agrega (append) a la misma tabla Delta
        write_deltalake(bronze_path, df_dia, mode=mode)
        print(f"  Bronze: {dia} — {n} filas ingestadas (mode={mode})")
    else:
        print(f"  [ref] write_deltalake('{bronze_path}', df, mode='{mode}')")

# Verificar
if DELTA_OK:
    dt_bronze = DeltaTable(bronze_path)
    df_bronze = dt_bronze.to_pandas()
    print(f"\nBronze total: {len(df_bronze)} filas, {len(dt_bronze.history())} versiones")

  [ref] write_deltalake('lakehouse/bronze/ventas', df, mode='overwrite')
  [ref] write_deltalake('lakehouse/bronze/ventas', df, mode='append')
  [ref] write_deltalake('lakehouse/bronze/ventas', df, mode='append')


In [6]:
# ============================================================
# VER LA ESTRUCTURA INTERNA DE UNA TABLA DELTA
# ============================================================

if DELTA_OK:
    print("Archivos dentro de la tabla Delta:\n")
    for root, dirs, files in os.walk(bronze_path):
        level = root.replace(bronze_path, '').count(os.sep)
        indent = '  ' * level
        dirname = os.path.basename(root)
        print(f'{indent}{dirname}/')
        subindent = '  ' * (level + 1)
        for f in sorted(files):
            size = os.path.getsize(os.path.join(root, f))
            tag = '← log de transacciones' if '_delta_log' in root else '← datos (Parquet)'
            print(f'{subindent}{f:55s} {size:>8,} bytes  {tag}')

    print("\nEl _delta_log/ es lo que hace Delta diferente de Parquet puro.")
    print("Controla que archivos son validos en cada version.")
else:
    print("Instala deltalake para ver la estructura.")

Instala deltalake para ver la estructura.


In [7]:
# ============================================================
# HISTORIAL DE VERSIONES
# ============================================================

if DELTA_OK:
    print("Historial de la tabla Bronze:\n")
    for entry in dt_bronze.history():
        version = entry.get('version', '?')
        op = entry.get('operation', '?')
        ts = entry.get('timestamp', '?')
        metrics = entry.get('operationMetrics', {})
        rows = metrics.get('numOutputRows', '?')
        files = metrics.get('numAddedFiles', '?')
        print(f"  Version {version}: {op:10s} | {rows:>4s} filas | {files} archivos | {ts}")

---
## 3. Schema enforcement

In [8]:
# ============================================================
# SCHEMA ENFORCEMENT: rechazar datos incompatibles
# ============================================================

# Intentar agregar datos con una columna extra
datos_malos = pd.DataFrame({
    'fecha': [pd.Timestamp('2024-06-04')],
    'producto': ['Dashboard'],
    'region': ['Bogota'],
    'cliente': ['NuevoCliente'],
    'unidades': [10],
    'precio': [450.0],
    'ingesta_timestamp': [datetime.now().isoformat()],
    'columna_extra': ['esto no deberia estar'],  # No esta en el esquema
})

if DELTA_OK:
    try:
        write_deltalake(bronze_path, datos_malos, mode='append')
        print("Escribio sin error (schema_mode default permite merge en algunas versiones)")
    except Exception as e:
        print(f"RECHAZADO (esperado): {type(e).__name__}")
        print(f"  {str(e)[:200]}")
        print("\nDelta protege la tabla: no permite agregar columnas sin permiso.")
else:
    print("[ref] Si la tabla tiene 7 columnas y el DataFrame tiene 8,")
    print("Delta rechaza la escritura por defecto.")

[ref] Si la tabla tiene 7 columnas y el DataFrame tiene 8,
Delta rechaza la escritura por defecto.


In [9]:
# ============================================================
# SCHEMA EVOLUTION: agregar columna de forma controlada
# ============================================================

# Cuando SI necesitas agregar una columna, lo haces explicitamente

if DELTA_OK:
    try:
        write_deltalake(
            bronze_path,
            datos_malos,
            mode='append',
            schema_mode='merge'  # Permite agregar columnas nuevas
        )
        dt_bronze = DeltaTable(bronze_path)
        print("Columna agregada con schema_mode='merge'")
        print(f"Esquema actual: {dt_bronze.schema().to_pyarrow()}")
        print(f"\nLas filas anteriores tienen NULL en 'columna_extra'.")
        print(f"Total filas: {len(dt_bronze.to_pandas())}")
    except Exception as e:
        print(f"Error: {e}")
else:
    print("[ref] write_deltalake(path, df, mode='append', schema_mode='merge')")
    print("Las filas viejas tendran NULL en la columna nueva.")

[ref] write_deltalake(path, df, mode='append', schema_mode='merge')
Las filas viejas tendran NULL en la columna nueva.


---
## 4. Time travel

In [10]:
# ============================================================
# LEER UNA VERSION ANTERIOR
# ============================================================

if DELTA_OK:
    dt = DeltaTable(bronze_path)
    version_actual = dt.version()
    df_actual = dt.to_pandas()

    print(f"Version actual: {version_actual} ({len(df_actual)} filas)")

    # Leer la version 0 (solo el primer dia)
    dt_v0 = DeltaTable(bronze_path, version=0)
    df_v0 = dt_v0.to_pandas()
    print(f"Version 0:      {0} ({len(df_v0)} filas)")

    # Leer la version 1 (primer + segundo dia)
    dt_v1 = DeltaTable(bronze_path, version=1)
    df_v1 = dt_v1.to_pandas()
    print(f"Version 1:      {1} ({len(df_v1)} filas)")

    print(f"\nCada version es una foto de la tabla en ese momento.")
    print(f"Si algo sale mal, puedes volver atras sin perder datos.")
else:
    print("[ref] DeltaTable(path, version=0)  # Leer version especifica")

[ref] DeltaTable(path, version=0)  # Leer version especifica


In [11]:
# ============================================================
# CASO DE USO: restaurar despues de un error
# ============================================================

if DELTA_OK:
    # Simular un error: sobreescribir con datos incorrectos
    datos_error = pd.DataFrame({
        'fecha': [pd.Timestamp('2024-01-01')],
        'producto': ['ERROR'],
        'region': ['ERROR'],
        'cliente': ['ERROR'],
        'unidades': [0],
        'precio': [0.0],
        'ingesta_timestamp': ['error'],
    })

    version_buena = dt.version()
    write_deltalake(bronze_path, datos_error, mode='overwrite')

    dt_error = DeltaTable(bronze_path)
    print(f"Despues del error: {len(dt_error.to_pandas())} fila (todo se perdio?)")
    print(f"Version actual: {dt_error.version()}")

    # Restaurar la version buena
    dt_buena = DeltaTable(bronze_path, version=version_buena)
    df_restaurado = dt_buena.to_pandas()

    # Reescribir con los datos buenos
    write_deltalake(bronze_path, df_restaurado, mode='overwrite')

    dt_final = DeltaTable(bronze_path)
    print(f"\nDespues de restaurar: {len(dt_final.to_pandas())} filas")
    print(f"Version actual: {dt_final.version()}")
    print(f"\nDatos recuperados. Sin time travel, se habrian perdido.")
else:
    print("[ref] Flujo de restauracion:")
    print("  1. Alguien hace overwrite con datos incorrectos")
    print("  2. La tabla ahora tiene 1 fila en vez de 300")
    print("  3. Leer la version anterior: DeltaTable(path, version=N)")
    print("  4. Reescribir con los datos buenos")
    print("  5. Todo restaurado")

[ref] Flujo de restauracion:
  1. Alguien hace overwrite con datos incorrectos
  2. La tabla ahora tiene 1 fila en vez de 300
  3. Leer la version anterior: DeltaTable(path, version=N)
  4. Reescribir con los datos buenos
  5. Todo restaurado


---
## 5. Merge / Upsert

In [12]:
# ============================================================
# CREAR UNA TABLA SILVER CON CLIENTES
# ============================================================

silver_clientes_path = f'{LAKEHOUSE}/silver/clientes'

clientes = pd.DataFrame({
    'cliente_id': [1, 2, 3, 4, 5],
    'nombre': ['TechCorp', 'DataSoft', 'Analitika', 'InfoSystems', 'CloudBI'],
    'segmento': ['startup', 'profesional', 'startup', 'enterprise', 'profesional'],
    'ciudad': ['medellin', 'bogota', 'cali', 'bogota', 'manizales'],
    'mrr': [500.0, 1500.0, 500.0, 5000.0, 1500.0],
    'actualizado': [datetime.now().isoformat()] * 5,
})

if DELTA_OK:
    write_deltalake(silver_clientes_path, clientes, mode='overwrite')
    print("Tabla Silver clientes creada:")
    print(DeltaTable(silver_clientes_path).to_pandas().to_string(index=False))
else:
    print(clientes.to_string(index=False))

 cliente_id      nombre    segmento    ciudad    mrr                actualizado
          1    TechCorp     startup  medellin  500.0 2026-09-12T02:23:58.994750
          2    DataSoft profesional    bogota 1500.0 2026-09-12T02:23:58.994750
          3   Analitika     startup      cali  500.0 2026-09-12T02:23:58.994750
          4 InfoSystems  enterprise    bogota 5000.0 2026-09-12T02:23:58.994750
          5     CloudBI profesional manizales 1500.0 2026-09-12T02:23:58.994750


In [13]:
# ============================================================
# MERGE: actualizar existentes + insertar nuevos
# ============================================================

# Llegan datos nuevos del CRM:
# - TechCorp cambio de segmento (startup → enterprise) y de ciudad
# - CloudBI aumento su MRR
# - VisionData es un cliente nuevo

datos_nuevos = pd.DataFrame({
    'cliente_id': [1, 5, 6],
    'nombre': ['TechCorp', 'CloudBI', 'VisionData'],
    'segmento': ['enterprise', 'profesional', 'startup'],
    'ciudad': ['bogota', 'manizales', 'barranquilla'],
    'mrr': [5000.0, 2200.0, 800.0],
    'actualizado': [datetime.now().isoformat()] * 3,
})

print("Datos nuevos del CRM:")
print(datos_nuevos.to_string(index=False))
print("\n  cliente_id=1 (TechCorp): actualizar segmento y ciudad")
print("  cliente_id=5 (CloudBI): actualizar MRR")
print("  cliente_id=6 (VisionData): insertar (nuevo)")

Datos nuevos del CRM:
 cliente_id     nombre    segmento       ciudad    mrr                actualizado
          1   TechCorp  enterprise       bogota 5000.0 2026-09-12T02:23:59.010410
          5    CloudBI profesional    manizales 2200.0 2026-09-12T02:23:59.010410
          6 VisionData     startup barranquilla  800.0 2026-09-12T02:23:59.010410

  cliente_id=1 (TechCorp): actualizar segmento y ciudad
  cliente_id=5 (CloudBI): actualizar MRR
  cliente_id=6 (VisionData): insertar (nuevo)


In [14]:
# ============================================================
# EJECUTAR EL MERGE
# ============================================================

if DELTA_OK:
    dt_clientes = DeltaTable(silver_clientes_path)

    (
        dt_clientes.merge(
            source=datos_nuevos,
            predicate='s.cliente_id = t.cliente_id',  # Condicion de match
            source_alias='s',
            target_alias='t',
        )
        .when_matched_update_all()   # Si existe: actualizar todos los campos
        .when_not_matched_insert_all()  # Si no existe: insertar
        .execute()
    )

    print("Despues del merge:\n")
    df_merged = DeltaTable(silver_clientes_path).to_pandas()
    print(df_merged.to_string(index=False))

    print(f"\nFilas: {len(df_merged)} (antes 5, ahora 6 — VisionData se inserto)")
    print(f"TechCorp: segmento={df_merged[df_merged['cliente_id']==1]['segmento'].iloc[0]}, "
          f"ciudad={df_merged[df_merged['cliente_id']==1]['ciudad'].iloc[0]} (actualizado)")
else:
    print("[ref] Merge con deltalake:")
    print("  dt.merge(source=df, predicate='s.id = t.id', ...)")
    print("    .when_matched_update_all()")
    print("    .when_not_matched_insert_all()")
    print("    .execute()")
    print("")
    print("  Si el id existe: actualiza todos los campos")
    print("  Si el id no existe: inserta la fila")
    print("  Todo en una operacion atomica")

[ref] Merge con deltalake:
  dt.merge(source=df, predicate='s.id = t.id', ...)
    .when_matched_update_all()
    .when_not_matched_insert_all()
    .execute()

  Si el id existe: actualiza todos los campos
  Si el id no existe: inserta la fila
  Todo en una operacion atomica


In [15]:
# ============================================================
# VERIFICAR HISTORIAL DEL MERGE
# ============================================================

if DELTA_OK:
    dt = DeltaTable(silver_clientes_path)
    print("Historial de la tabla clientes:\n")
    for entry in dt.history():
        v = entry.get('version', '?')
        op = entry.get('operation', '?')
        metrics = entry.get('operationMetrics', {})
        print(f"  Version {v}: {op}")
        if 'numTargetRowsUpdated' in metrics:
            print(f"    Actualizadas: {metrics.get('numTargetRowsUpdated', 0)}")
            print(f"    Insertadas: {metrics.get('numTargetRowsInserted', 0)}")

---
## 6. Medallion Architecture completa con Delta

In [16]:
# ============================================================
# PIPELINE COMPLETO: Bronze → Silver → Gold
# ============================================================

def pipeline_bronze_a_silver(lakehouse_path):
    """
    Lee los datos crudos de Bronze, limpia y guarda en Silver.
    """
    bronze = f'{lakehouse_path}/bronze/ventas'
    silver = f'{lakehouse_path}/silver/ventas'

    # Leer Bronze
    dt = DeltaTable(bronze)
    df = dt.to_pandas()
    print(f"  Bronze: {len(df)} filas leidas")

    # Transformar
    df['producto'] = df['producto'].str.strip().str.lower()
    df['region'] = df['region'].str.strip().str.lower()
    df['cliente'] = df['cliente'].str.strip().str.lower()
    df['ingreso'] = df['unidades'] * df['precio']

    # Quitar la columna de metadata de ingesta y columnas extras
    cols_silver = ['fecha', 'producto', 'region', 'cliente', 'unidades', 'precio', 'ingreso']
    cols_disponibles = [c for c in cols_silver if c in df.columns]
    df = df[cols_disponibles]

    # Deduplicar
    antes = len(df)
    df = df.drop_duplicates()
    print(f"  Deduplicacion: {antes} → {len(df)} filas")

    # Escribir Silver
    write_deltalake(silver, df, mode='overwrite')
    print(f"  Silver: {len(df)} filas escritas")

    return df

def pipeline_silver_a_gold(lakehouse_path):
    """
    Lee Silver, agrega y guarda en Gold para dashboards.
    """
    silver = f'{lakehouse_path}/silver/ventas'
    gold_regional = f'{lakehouse_path}/gold/resumen_regional'
    gold_producto = f'{lakehouse_path}/gold/resumen_producto'
    gold_diario = f'{lakehouse_path}/gold/kpis_diarios'

    # Leer Silver
    df = DeltaTable(silver).to_pandas()
    print(f"  Silver: {len(df)} filas leidas")

    # Gold: resumen regional
    resumen_reg = (df.groupby('region')
        .agg(ingreso=('ingreso', 'sum'), transacciones=('ingreso', 'count'),
             ticket_promedio=('ingreso', 'mean'))
        .round(2).reset_index()
        .sort_values('ingreso', ascending=False))
    write_deltalake(gold_regional, resumen_reg, mode='overwrite')
    print(f"  Gold resumen_regional: {len(resumen_reg)} filas")

    # Gold: resumen por producto
    resumen_prod = (df.groupby('producto')
        .agg(ingreso=('ingreso', 'sum'), transacciones=('ingreso', 'count'),
             precio_promedio=('precio', 'mean'))
        .round(2).reset_index()
        .sort_values('ingreso', ascending=False))
    write_deltalake(gold_producto, resumen_prod, mode='overwrite')
    print(f"  Gold resumen_producto: {len(resumen_prod)} filas")

    # Gold: KPIs diarios
    kpis = (df.groupby('fecha')
        .agg(ingreso=('ingreso', 'sum'), transacciones=('ingreso', 'count'),
             clientes=('cliente', 'nunique'))
        .round(2).reset_index())
    write_deltalake(gold_diario, kpis, mode='overwrite')
    print(f"  Gold kpis_diarios: {len(kpis)} filas")

    return resumen_reg, resumen_prod, kpis

if DELTA_OK:
    print("=" * 50)
    print("  PIPELINE: Bronze → Silver")
    print("=" * 50)
    df_silver = pipeline_bronze_a_silver(LAKEHOUSE)

    print(f"\n{'=' * 50}")
    print("  PIPELINE: Silver → Gold")
    print("=" * 50)
    reg, prod, kpis = pipeline_silver_a_gold(LAKEHOUSE)
else:
    print("[ref] El pipeline lee Delta, transforma, y escribe Delta.")
    print("Cada zona es una tabla con historial y esquema.")

[ref] El pipeline lee Delta, transforma, y escribe Delta.
Cada zona es una tabla con historial y esquema.


In [17]:
# ============================================================
# RESULTADO: tablas Gold listas para consumo
# ============================================================

if DELTA_OK:
    print("=== Gold: Resumen Regional ===")
    print(DeltaTable(f'{LAKEHOUSE}/gold/resumen_regional').to_pandas().to_string(index=False))

    print("\n=== Gold: Resumen Producto ===")
    print(DeltaTable(f'{LAKEHOUSE}/gold/resumen_producto').to_pandas().to_string(index=False))

    print("\n=== Gold: KPIs Diarios ===")
    print(DeltaTable(f'{LAKEHOUSE}/gold/kpis_diarios').to_pandas().to_string(index=False))

---
## 7. Consultas analiticas sobre Delta

In [18]:
# ============================================================
# CONSULTAR CON PANDAS (sin Spark)
# ============================================================

if DELTA_OK:
    # Leer Silver y hacer analisis
    df = DeltaTable(f'{LAKEHOUSE}/silver/ventas').to_pandas()

    # Top 3 clientes por ingreso
    print("Top 3 clientes:")
    top_clientes = (df.groupby('cliente')['ingreso']
                    .sum()
                    .sort_values(ascending=False)
                    .head(3))
    print(top_clientes.to_string())

    # Ticket promedio por region
    print("\nTicket promedio por region:")
    ticket = (df.groupby('region')['ingreso']
              .mean()
              .round(2)
              .sort_values(ascending=False))
    print(ticket.to_string())

In [19]:
# ============================================================
# CONSULTAR CON DUCKDB (SQL sobre Delta, sin Spark)
# ============================================================

# pip install duckdb

try:
    import duckdb

    # DuckDB puede leer tablas Delta directamente con SQL
    result = duckdb.sql(f"""
        SELECT
            region,
            producto,
            SUM(ingreso) AS ingreso_total,
            COUNT(*) AS transacciones
        FROM delta_scan('{LAKEHOUSE}/silver/ventas')
        GROUP BY region, producto
        ORDER BY ingreso_total DESC
        LIMIT 10
    """).df()

    print("SQL con DuckDB sobre tabla Delta (sin Spark):\n")
    print(result.to_string(index=False))

except ImportError:
    print("DuckDB no instalado. Ejecuta: pip install duckdb")
    print("Permite hacer consultas SQL sobre tablas Delta sin Spark.")
except Exception as e:
    print(f"Error: {e}")
    print("Puede requerir: pip install duckdb --upgrade")

Error: IO Error: DeltKernel InvalidTableLocationError (28): Invalid table location: Path does not exist: "lakehouse/silver/ventas/".
Puede requerir: pip install duckdb --upgrade


In [20]:
# ============================================================
# VISTA COMPLETA DEL LAKEHOUSE
# ============================================================

if DELTA_OK:
    print("Estado del Lakehouse:\n")

    tablas = [
        ('bronze', 'ventas', f'{LAKEHOUSE}/bronze/ventas'),
        ('silver', 'ventas', f'{LAKEHOUSE}/silver/ventas'),
        ('silver', 'clientes', f'{LAKEHOUSE}/silver/clientes'),
        ('gold', 'resumen_regional', f'{LAKEHOUSE}/gold/resumen_regional'),
        ('gold', 'resumen_producto', f'{LAKEHOUSE}/gold/resumen_producto'),
        ('gold', 'kpis_diarios', f'{LAKEHOUSE}/gold/kpis_diarios'),
    ]

    for zona, nombre, path in tablas:
        if os.path.exists(path):
            dt = DeltaTable(path)
            n_filas = len(dt.to_pandas())
            n_versiones = len(dt.history())
            n_cols = len(dt.schema().to_pyarrow())
            print(f"  [{zona:6s}] {nombre:25s} {n_filas:>6,} filas  {n_cols} cols  {n_versiones} versiones")

In [21]:
# Limpiar
if os.path.exists(LAKEHOUSE):
    shutil.rmtree(LAKEHOUSE)
    print("Lakehouse eliminado.")

Lakehouse eliminado.


---
## Resumen

| Concepto | Lo que importa |
|---|---|
| **Lakehouse** | Almacenamiento barato (archivos) + capa de metadata (Delta) = Warehouse + Lake en uno |
| **Delta Lake** | La capa que agrega ACID, time travel, esquema y merge sobre Parquet |
| **Schema enforcement** | Rechaza datos que no cumplen el esquema. Previene corrupcion |
| **Schema evolution** | Agregar columnas de forma controlada con `schema_mode='merge'` |
| **Time travel** | Leer cualquier version anterior. Restaurar despues de un error |
| **Merge/Upsert** | Actualizar existentes + insertar nuevos en una operacion atomica |
| **Medallion** | Bronze (Delta append-only) → Silver (Delta con esquema) → Gold (Delta agregado) |
| **Sin Spark** | La libreria `deltalake` permite todo esto en Python puro |

### La diferencia clave

Un Data Lake es carpetas con archivos. Un Lakehouse es **tablas** con archivos. La tabla tiene esquema, historial, transacciones. La carpeta no.

### Siguiente paso
En el **Notebook 4.2** construimos pipelines ETL/ELT completos para mover datos entre las zonas del Lakehouse.